# 01. Classes, objects, self, and constructors

**Level:** Beginner

**Learning outcome:** Create a class, trace initialization, distinguish identity from value, and choose queries versus state changes.

**Prerequisites:** Basic functions, variables, dictionaries, and indentation.

[Learning-path index](00_START_HERE_OOP_Learning_Path.ipynb) | [Previous](00_START_HERE_OOP_Learning_Path.ipynb) | [Next notebook](02_Attributes_Method_Types_and_Safe_Defaults.ipynb)

## How to study this notebook
1. Read the mental model and explanation before executing a code cell.
2. Predict the output, then run the cell and compare the stored result.
3. Trace constructor calls, attribute reads/writes, returned values, and exception paths.
4. Change one input, predict the difference, and keep meaningful assertions.
5. Complete the practice prompts and answer the final checkpoint aloud.

**Run independently:** every worked code cell includes its own imports, definitions, and input data. You do not need to run another notebook first. Use Python 3.11+ and a Python Jupyter kernel. Only the standard library is used. Async cells use top-level await, not asyncio.run inside Jupyter.

**Offline safety:** all network states, commands, approvals, and failures are simulations. The code does not connect to devices, use credentials, write device configuration, or need external input files. Teaching limits and command strings are not vendor specifications. Assertions are learning checks, not production input validation.

## Concept notes and mental model

**Level:** Beginner

**Learning goal:** Read a class definition, build independent objects, and trace changes to instance state.

**What OOP means.** Object-oriented programming organizes a program around objects that combine
information (attributes) with operations (methods). A network device is a useful model: it has
a hostname and interface state, and it can produce a description or validate a proposed change.
An object here represents a device; it is not the physical device and cannot reach one unless
you deliberately give it an I/O implementation. These lessons only use in-memory simulations.

**Essential vocabulary.** A class defines a type and its behavior. An instance is one object of
that type. An attribute is a name accessed with a dot. A method is a function obtained through
an object or class. State is the collection of current values. An invariant is a rule that must
remain true, such as a nonnegative counter. An interface is the public contract callers use.

**Read the syntax slowly.** `class Device:` starts an indented class body. A `def` within it
defines a method. Calling `Device("edge-1")` creates an instance and normally initializes it
through `__init__`. More precisely, `__new__` creates the instance and `__init__` initializes
it; `__init__` must return `None`. `self` is the conventional name for the instance passed to a
bound method. It is not a keyword. `self.hostname = hostname` stores the argument on that object.
`return` hands a value to the caller; `print` displays a value. They are not interchangeable.

**A trace to keep in mind.** Create an object -> initialize its fields -> call a method -> read
or update its state -> return a result -> check the result. Two calls to a class normally
create two independent instances. Assigning an existing object to another variable does not
create a copy; it creates another reference to the same object.

**When a class helps.** Use one when a concept has meaningful state, invariants, interchangeable
behavior, or a resource lifetime. A one-line conversion usually deserves a plain function, not
a class with no purpose. Start with a small model, not a hierarchy of Router/Platform/Region/
Organization subclasses. Model the behavior the automation actually needs.

**Pause and explain:** Explain class versus instance; trace `self`; distinguish a returned result from printed output.


## Slow walkthrough: how to read your first class

Use Example 0001 as your reference. Do not try to memorize every special method before you understand
what happens in this small program.

1. **`class Device:`** executes a class definition and binds the resulting class object to the name
   Device. Indentation places the following functions in that class's namespace. The class is not a
   physical network device and defining it does not create a connection.
2. **`def __init__(self, hostname):`** defines the initializer. After a normal instance is created,
   Python calls this method with the instance and your supplied hostname. You do not supply self
   separately when calling Device(...). The object may receive many initial values; this first
   example intentionally starts with just one.
3. **`self.hostname = hostname`** reads the parameter on the right and stores it as an attribute on
   the instance on the left. The two uses of hostname have different roles. Calling the parameter
   name something else would work too; consistent naming simply makes the relationship clear.
4. **`def label(self):`** defines an instance method that needs no additional explicit arguments.
   It can read the instance's hostname through self. Local variables from an earlier method call are
   not automatically available here.
5. **`return f"device:{self.hostname}"`** produces a new string. The f-string evaluates the expression
   inside braces. The return exits this method and sends the string to the caller; it does not itself
   display anything.
6. **`device = Device(case["hostname"])`** first looks up a string in the local input dictionary, then
   calls the class, then binds the resulting instance to the variable device. The lowercase variable
   name and the uppercase class name are a convention, not two unrelated kinds of syntax.
7. **`device.label()`** retrieves a bound method and calls it. Python supplies device as self. The
   same class can serve many objects because each call uses the particular instance's attributes.
8. **`assert ...`** compares the actual returned value with the expected one. A false assertion raises
   AssertionError. It is an executable learning check, not an operational input-validation mechanism.
9. **`print(device.label())`** calls the method again and displays its returned string. Nothing about
   that print changes the stored hostname. A method that prints instead of returning would return
   None unless it has an explicit return statement.

### Follow state, not just syntax
Before construction there is no device instance in this example. After construction, its hostname is
the input string. Calling label reads that state without changing it. By contrast, the Interface
shutdown example intentionally changes admin_up from True to False. Classify each method you write as
a query, a command that mutates state, a conversion, or an I/O boundary. Clear intent is more valuable
than making every method look alike.

### Procedural code versus an object model
A dictionary plus a function can solve the first example. OOP becomes useful when related operations
share state and must protect common rules: constructing an interface, changing its MTU, rendering its
desired state, and checking a postcondition all need consistent values. Do not wrap every standalone
function in a class merely to call the result object-oriented. It is normal for good Python programs
to combine functions, classes, comprehensions, and immutable records.

### Four pillars in one simulated change
Encapsulation validates a proposed value before storing it. Abstraction lets a service call a transport
without knowing its library. Inheritance can specialize a true device subtype. Polymorphism lets the
same service call different compatible transports. Composition connects a service, a policy, and a
transport without requiring one subclass for every combination. None of these ideas requires
credentials or live I/O to learn and test.

## Design principles: use these as questions, not slogans

| Principle | Question to ask in a network automation review | Concrete application |
| --- | --- | --- |
| Single responsibility | Does this class have one coherent reason to change? | Separate inventory parsing, transport, policy, and reporting |
| Open/closed | Can a new behavior be supplied without editing a stable client? | Inject a new policy implementation into the existing evaluator |
| Liskov substitution | Can this child honor what callers of the base were promised? | Do not make a read-only reader mutate configuration |
| Interface segregation | Does a client depend only on operations it needs? | A Collector needs read/send, not deployment and billing methods |
| Dependency inversion | Does orchestration depend on a small contract instead of a concrete library? | Type against a Transport protocol and supply an adapter at startup |

These are the SOLID principles in practical form. They do not require five extra classes in every
script. A function can satisfy a dependency; a dataclass can be a value without a repository; two
well-named methods may be clearer than a generalized framework. A good abstraction removes real
duplication or isolates a real change boundary. An abstraction that has only speculative future uses
adds cognitive cost without present benefit.

### Equality, hashing, and operator decisions
Use `is` to ask whether two references point to the same object, and `==` to ask whether two values
compare equal. The frozen RouteKey example demonstrates equal independently created values, matching
hashes, and set deduplication. Ordinary mutable dataclasses are generally not hashable by default
because mutation could invalidate a hash-based collection's assumptions.

If you write `__eq__` manually, compare the intended domain key and normally return NotImplemented
for an unsupported other type. NotImplemented is a protocol signal, not the NotImplementedError
exception. If you supply `__hash__`, equal values must have equal hashes and the compared key must
remain stable while stored in a set or dictionary. Do not use a hash as a unique identifier or assume
its value is stable across Python processes.

Operator overloading gives syntax such as addition meaning through methods such as `__add__` and
`__radd__`. Use it only for natural domain arithmetic with defined units and compatible types:
adding two compatible traffic totals could make sense, while adding two live SSH sessions probably
does not. Return a new value rather than surprising callers by mutating an operand. Prefer a named
method when a symbol would hide units, network scope, or operational side effects. `__call__` instead
lets an instance be invoked like a function; the backoff policy shows this directly with policy(index).

### Overloading, overriding, and constructors are different
Overriding changes an inherited method in a subclass. Operator overloading implements a special
method for Python syntax. Signature-based method overloading, as seen in some other languages, is not
how ordinary Python class definitions work: a second same-named def replaces the earlier binding.
Use defaults or distinct method names, and remember that typing overload declarations are static
descriptions, not multiple runtime implementations. Named classmethod constructors express multiple
ways to build an object without redefining __init__ repeatedly.

### Lifecycle and errors
Creating an object should normally validate its configuration, not secretly perform a slow login.
Make external actions explicit. Resource cleanup belongs in with/async with or a deliberate close
method. An object's destructor is not a dependable place to commit changes or close a production
session, because finalization timing and shutdown conditions can differ.

Distinguish programming errors, invalid input, policy rejection, transient transport failure, and
failed postchecks. A message that says success while returning no data after a timeout is misleading.
These examples display expected rejections explicitly and assert the failure behavior. Their fakes
cannot prove a real device supports a command or that a real rollout is safe.


## Worked-example structure

This notebook contains **5 worked patterns and 50 executable scenario examples**. Each pattern explains its purpose, logic, common mistakes, and extension exercise. Its ten scenarios vary the input data; they are related practice, not ten unrelated algorithms. Original Example IDs are retained so you can locate the same exercise in `AI_oops_learning.ipynb`.

The input table lists only fields needed by that pattern. Expected output is shown before each code cell and a saved executed result appears afterward. Expected rejection branches are checked explicitly; unexpected exceptions are not silently ignored.

## Pattern 001: Create a device instance

**Core concept:** A class bundles identity and a small public operation.

**Network-engineering purpose:** An inventory object gives later checks one consistent hostname instead of unrelated variables.

### Read the implementation in this order
1. Define Device before calling it.
2. Store hostname on self in __init__.
3. Call label() through the instance; Python supplies self.
4. Assert the returned string and print it.

**Common mistake / boundary:** Leaving off self when assigning hostname creates only a temporary local variable.

**Practice after the ten worked scenarios:** Add a role field and include it in label() without adding a second class.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0001 - Create a device instance: branch scenario

**Scenario 01/10; pattern 001; Beginner.** This is a self-contained offline exercise. An inventory object gives later checks one consistent hostname instead of unrelated variables.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Define Device before calling it. Assert the returned string and print it. Follow the class-specific steps under Pattern 001 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
device:branch-edge-01
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Leaving off self when assigning hostname creates only a temporary local variable.

In [1]:
# Example 0001: Create a device instance - branch
case = {'hostname': 'branch-edge-01'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

    def label(self):
        return f"device:{self.hostname}"

device = Device(case["hostname"])
assert device.label() == "device:" + case["hostname"]
print(device.label())


device:branch-edge-01


### Example 0002 - Create a device instance: campus scenario

**Scenario 02/10; pattern 001; Beginner.** This is a self-contained offline exercise. An inventory object gives later checks one consistent hostname instead of unrelated variables.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Define Device before calling it. Assert the returned string and print it. Follow the class-specific steps under Pattern 001 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
device:campus-access-02
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Leaving off self when assigning hostname creates only a temporary local variable.

In [2]:
# Example 0002: Create a device instance - campus
case = {'hostname': 'campus-access-02'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

    def label(self):
        return f"device:{self.hostname}"

device = Device(case["hostname"])
assert device.label() == "device:" + case["hostname"]
print(device.label())


device:campus-access-02


### Example 0003 - Create a device instance: dc scenario

**Scenario 03/10; pattern 001; Beginner.** This is a self-contained offline exercise. An inventory object gives later checks one consistent hostname instead of unrelated variables.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Define Device before calling it. Assert the returned string and print it. Follow the class-specific steps under Pattern 001 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
device:dc-leaf-03
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Leaving off self when assigning hostname creates only a temporary local variable.

In [3]:
# Example 0003: Create a device instance - dc
case = {'hostname': 'dc-leaf-03'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

    def label(self):
        return f"device:{self.hostname}"

device = Device(case["hostname"])
assert device.label() == "device:" + case["hostname"]
print(device.label())


device:dc-leaf-03


### Example 0004 - Create a device instance: wan scenario

**Scenario 04/10; pattern 001; Beginner.** This is a self-contained offline exercise. An inventory object gives later checks one consistent hostname instead of unrelated variables.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Define Device before calling it. Assert the returned string and print it. Follow the class-specific steps under Pattern 001 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
device:wan-border-04
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Leaving off self when assigning hostname creates only a temporary local variable.

In [4]:
# Example 0004: Create a device instance - wan
case = {'hostname': 'wan-border-04'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

    def label(self):
        return f"device:{self.hostname}"

device = Device(case["hostname"])
assert device.label() == "device:" + case["hostname"]
print(device.label())


device:wan-border-04


### Example 0005 - Create a device instance: lab scenario

**Scenario 05/10; pattern 001; Beginner.** This is a self-contained offline exercise. An inventory object gives later checks one consistent hostname instead of unrelated variables.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Define Device before calling it. Assert the returned string and print it. Follow the class-specific steps under Pattern 001 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
device:lab-router-05
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Leaving off self when assigning hostname creates only a temporary local variable.

In [5]:
# Example 0005: Create a device instance - lab
case = {'hostname': 'lab-router-05'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

    def label(self):
        return f"device:{self.hostname}"

device = Device(case["hostname"])
assert device.label() == "device:" + case["hostname"]
print(device.label())


device:lab-router-05


### Example 0006 - Create a device instance: wireless scenario

**Scenario 06/10; pattern 001; Beginner.** This is a self-contained offline exercise. An inventory object gives later checks one consistent hostname instead of unrelated variables.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Define Device before calling it. Assert the returned string and print it. Follow the class-specific steps under Pattern 001 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
device:wireless-dist-06
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Leaving off self when assigning hostname creates only a temporary local variable.

In [6]:
# Example 0006: Create a device instance - wireless
case = {'hostname': 'wireless-dist-06'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

    def label(self):
        return f"device:{self.hostname}"

device = Device(case["hostname"])
assert device.label() == "device:" + case["hostname"]
print(device.label())


device:wireless-dist-06


### Example 0007 - Create a device instance: remote scenario

**Scenario 07/10; pattern 001; Beginner.** This is a self-contained offline exercise. An inventory object gives later checks one consistent hostname instead of unrelated variables.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Define Device before calling it. Assert the returned string and print it. Follow the class-specific steps under Pattern 001 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
device:remote-gateway-07
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Leaving off self when assigning hostname creates only a temporary local variable.

In [7]:
# Example 0007: Create a device instance - remote
case = {'hostname': 'remote-gateway-07'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

    def label(self):
        return f"device:{self.hostname}"

device = Device(case["hostname"])
assert device.label() == "device:" + case["hostname"]
print(device.label())


device:remote-gateway-07


### Example 0008 - Create a device instance: core scenario

**Scenario 08/10; pattern 001; Beginner.** This is a self-contained offline exercise. An inventory object gives later checks one consistent hostname instead of unrelated variables.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Define Device before calling it. Assert the returned string and print it. Follow the class-specific steps under Pattern 001 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
device:core-router-08
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Leaving off self when assigning hostname creates only a temporary local variable.

In [8]:
# Example 0008: Create a device instance - core
case = {'hostname': 'core-router-08'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

    def label(self):
        return f"device:{self.hostname}"

device = Device(case["hostname"])
assert device.label() == "device:" + case["hostname"]
print(device.label())


device:core-router-08


### Example 0009 - Create a device instance: mgmt scenario

**Scenario 09/10; pattern 001; Beginner.** This is a self-contained offline exercise. An inventory object gives later checks one consistent hostname instead of unrelated variables.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Define Device before calling it. Assert the returned string and print it. Follow the class-specific steps under Pattern 001 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
device:mgmt-switch-09
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Leaving off self when assigning hostname creates only a temporary local variable.

In [9]:
# Example 0009: Create a device instance - mgmt
case = {'hostname': 'mgmt-switch-09'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

    def label(self):
        return f"device:{self.hostname}"

device = Device(case["hostname"])
assert device.label() == "device:" + case["hostname"]
print(device.label())


device:mgmt-switch-09


### Example 0010 - Create a device instance: backup scenario

**Scenario 10/10; pattern 001; Beginner.** This is a self-contained offline exercise. An inventory object gives later checks one consistent hostname instead of unrelated variables.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Define Device before calling it. Assert the returned string and print it. Follow the class-specific steps under Pattern 001 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
device:backup-edge-10
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Leaving off self when assigning hostname creates only a temporary local variable.

In [10]:
# Example 0010: Create a device instance - backup
case = {'hostname': 'backup-edge-10'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

    def label(self):
        return f"device:{self.hostname}"

device = Device(case["hostname"])
assert device.label() == "device:" + case["hostname"]
print(device.label())


device:backup-edge-10


## Pattern 002: Constructor arguments and defaults

**Core concept:** Initialization can provide a deliberate default while accepting overrides.

**Network-engineering purpose:** A simulated session records a port without opening a connection.

### Read the implementation in this order
1. Accept hostname and an optional port.
2. Initialize both fields once.
3. Build one object with the default and one with an explicit port.
4. Check that overriding one object does not affect another.

**Common mistake / boundary:** A default is a policy choice, not proof that a remote service exists or is reachable.

**Practice after the ten worked scenarios:** Reject ports outside 1..65535 before storing them.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0011 - Constructor arguments and defaults: branch scenario

**Scenario 01/10; pattern 002; Beginner.** This is a self-contained offline exercise. A simulated session records a port without opening a connection.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |
| `port` | `22` |

**Logic to trace:** Accept hostname and an optional port. Check that overriding one object does not affect another. Follow the class-specific steps under Pattern 002 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
22 22
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A default is a policy choice, not proof that a remote service exists or is reachable.

In [11]:
# Example 0011: Constructor arguments and defaults - branch
case = {'hostname': 'branch-edge-01', 'port': 22}

class Endpoint:
    def __init__(self, hostname, port=22):
        self.hostname = hostname
        self.port = port

default = Endpoint(case["hostname"])
custom = Endpoint(case["hostname"], case["port"])
assert default.port == 22
assert custom.port == case["port"]
print(default.port, custom.port)


22 22


### Example 0012 - Constructor arguments and defaults: campus scenario

**Scenario 02/10; pattern 002; Beginner.** This is a self-contained offline exercise. A simulated session records a port without opening a connection.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |
| `port` | `2222` |

**Logic to trace:** Accept hostname and an optional port. Check that overriding one object does not affect another. Follow the class-specific steps under Pattern 002 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
22 2222
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A default is a policy choice, not proof that a remote service exists or is reachable.

In [12]:
# Example 0012: Constructor arguments and defaults - campus
case = {'hostname': 'campus-access-02', 'port': 2222}

class Endpoint:
    def __init__(self, hostname, port=22):
        self.hostname = hostname
        self.port = port

default = Endpoint(case["hostname"])
custom = Endpoint(case["hostname"], case["port"])
assert default.port == 22
assert custom.port == case["port"]
print(default.port, custom.port)


22 2222


### Example 0013 - Constructor arguments and defaults: dc scenario

**Scenario 03/10; pattern 002; Beginner.** This is a self-contained offline exercise. A simulated session records a port without opening a connection.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |
| `port` | `830` |

**Logic to trace:** Accept hostname and an optional port. Check that overriding one object does not affect another. Follow the class-specific steps under Pattern 002 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
22 830
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A default is a policy choice, not proof that a remote service exists or is reachable.

In [13]:
# Example 0013: Constructor arguments and defaults - dc
case = {'hostname': 'dc-leaf-03', 'port': 830}

class Endpoint:
    def __init__(self, hostname, port=22):
        self.hostname = hostname
        self.port = port

default = Endpoint(case["hostname"])
custom = Endpoint(case["hostname"], case["port"])
assert default.port == 22
assert custom.port == case["port"]
print(default.port, custom.port)


22 830


### Example 0014 - Constructor arguments and defaults: wan scenario

**Scenario 04/10; pattern 002; Beginner.** This is a self-contained offline exercise. A simulated session records a port without opening a connection.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |
| `port` | `443` |

**Logic to trace:** Accept hostname and an optional port. Check that overriding one object does not affect another. Follow the class-specific steps under Pattern 002 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
22 443
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A default is a policy choice, not proof that a remote service exists or is reachable.

In [14]:
# Example 0014: Constructor arguments and defaults - wan
case = {'hostname': 'wan-border-04', 'port': 443}

class Endpoint:
    def __init__(self, hostname, port=22):
        self.hostname = hostname
        self.port = port

default = Endpoint(case["hostname"])
custom = Endpoint(case["hostname"], case["port"])
assert default.port == 22
assert custom.port == case["port"]
print(default.port, custom.port)


22 443


### Example 0015 - Constructor arguments and defaults: lab scenario

**Scenario 05/10; pattern 002; Beginner.** This is a self-contained offline exercise. A simulated session records a port without opening a connection.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |
| `port` | `2022` |

**Logic to trace:** Accept hostname and an optional port. Check that overriding one object does not affect another. Follow the class-specific steps under Pattern 002 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
22 2022
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A default is a policy choice, not proof that a remote service exists or is reachable.

In [15]:
# Example 0015: Constructor arguments and defaults - lab
case = {'hostname': 'lab-router-05', 'port': 2022}

class Endpoint:
    def __init__(self, hostname, port=22):
        self.hostname = hostname
        self.port = port

default = Endpoint(case["hostname"])
custom = Endpoint(case["hostname"], case["port"])
assert default.port == 22
assert custom.port == case["port"]
print(default.port, custom.port)


22 2022


### Example 0016 - Constructor arguments and defaults: wireless scenario

**Scenario 06/10; pattern 002; Beginner.** This is a self-contained offline exercise. A simulated session records a port without opening a connection.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |
| `port` | `22` |

**Logic to trace:** Accept hostname and an optional port. Check that overriding one object does not affect another. Follow the class-specific steps under Pattern 002 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
22 22
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A default is a policy choice, not proof that a remote service exists or is reachable.

In [16]:
# Example 0016: Constructor arguments and defaults - wireless
case = {'hostname': 'wireless-dist-06', 'port': 22}

class Endpoint:
    def __init__(self, hostname, port=22):
        self.hostname = hostname
        self.port = port

default = Endpoint(case["hostname"])
custom = Endpoint(case["hostname"], case["port"])
assert default.port == 22
assert custom.port == case["port"]
print(default.port, custom.port)


22 22


### Example 0017 - Constructor arguments and defaults: remote scenario

**Scenario 07/10; pattern 002; Beginner.** This is a self-contained offline exercise. A simulated session records a port without opening a connection.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |
| `port` | `8443` |

**Logic to trace:** Accept hostname and an optional port. Check that overriding one object does not affect another. Follow the class-specific steps under Pattern 002 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
22 8443
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A default is a policy choice, not proof that a remote service exists or is reachable.

In [17]:
# Example 0017: Constructor arguments and defaults - remote
case = {'hostname': 'remote-gateway-07', 'port': 8443}

class Endpoint:
    def __init__(self, hostname, port=22):
        self.hostname = hostname
        self.port = port

default = Endpoint(case["hostname"])
custom = Endpoint(case["hostname"], case["port"])
assert default.port == 22
assert custom.port == case["port"]
print(default.port, custom.port)


22 8443


### Example 0018 - Constructor arguments and defaults: core scenario

**Scenario 08/10; pattern 002; Beginner.** This is a self-contained offline exercise. A simulated session records a port without opening a connection.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |
| `port` | `830` |

**Logic to trace:** Accept hostname and an optional port. Check that overriding one object does not affect another. Follow the class-specific steps under Pattern 002 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
22 830
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A default is a policy choice, not proof that a remote service exists or is reachable.

In [18]:
# Example 0018: Constructor arguments and defaults - core
case = {'hostname': 'core-router-08', 'port': 830}

class Endpoint:
    def __init__(self, hostname, port=22):
        self.hostname = hostname
        self.port = port

default = Endpoint(case["hostname"])
custom = Endpoint(case["hostname"], case["port"])
assert default.port == 22
assert custom.port == case["port"]
print(default.port, custom.port)


22 830


### Example 0019 - Constructor arguments and defaults: mgmt scenario

**Scenario 09/10; pattern 002; Beginner.** This is a self-contained offline exercise. A simulated session records a port without opening a connection.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |
| `port` | `443` |

**Logic to trace:** Accept hostname and an optional port. Check that overriding one object does not affect another. Follow the class-specific steps under Pattern 002 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
22 443
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A default is a policy choice, not proof that a remote service exists or is reachable.

In [19]:
# Example 0019: Constructor arguments and defaults - mgmt
case = {'hostname': 'mgmt-switch-09', 'port': 443}

class Endpoint:
    def __init__(self, hostname, port=22):
        self.hostname = hostname
        self.port = port

default = Endpoint(case["hostname"])
custom = Endpoint(case["hostname"], case["port"])
assert default.port == 22
assert custom.port == case["port"]
print(default.port, custom.port)


22 443


### Example 0020 - Constructor arguments and defaults: backup scenario

**Scenario 10/10; pattern 002; Beginner.** This is a self-contained offline exercise. A simulated session records a port without opening a connection.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |
| `port` | `2222` |

**Logic to trace:** Accept hostname and an optional port. Check that overriding one object does not affect another. Follow the class-specific steps under Pattern 002 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
22 2222
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A default is a policy choice, not proof that a remote service exists or is reachable.

In [20]:
# Example 0020: Constructor arguments and defaults - backup
case = {'hostname': 'backup-edge-10', 'port': 2222}

class Endpoint:
    def __init__(self, hostname, port=22):
        self.hostname = hostname
        self.port = port

default = Endpoint(case["hostname"])
custom = Endpoint(case["hostname"], case["port"])
assert default.port == 22
assert custom.port == case["port"]
print(default.port, custom.port)


22 2222


## Pattern 003: An instance method changes state

**Core concept:** Methods can enforce a controlled state transition.

**Network-engineering purpose:** An administrative state model separates intended shutdown state from real operational telemetry.

### Read the implementation in this order
1. Initialize the interface as administratively up.
2. Use shutdown() to update that instance.
3. Return the new state for a caller to inspect.
4. Check the state before and after the operation.

**Common mistake / boundary:** Changing this Python attribute does not configure a switch; admin state is also not link state.

**Practice after the ten worked scenarios:** Add no_shutdown() and demonstrate that repeated calls are idempotent.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0021 - An instance method changes state: branch scenario

**Scenario 01/10; pattern 003; Beginner.** This is a self-contained offline exercise. An administrative state model separates intended shutdown state from real operational telemetry.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet1/1'` |

**Logic to trace:** Initialize the interface as administratively up. Check the state before and after the operation. Follow the class-specific steps under Pattern 003 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
Ethernet1/1 False
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Changing this Python attribute does not configure a switch; admin state is also not link state.

In [21]:
# Example 0021: An instance method changes state - branch
case = {'interface': 'Ethernet1/1'}

class Interface:
    def __init__(self, name):
        self.name = name
        self.admin_up = True

    def shutdown(self):
        self.admin_up = False
        return self.admin_up

interface = Interface(case["interface"])
assert interface.admin_up is True
assert interface.shutdown() is False
print(interface.name, interface.admin_up)


Ethernet1/1 False


### Example 0022 - An instance method changes state: campus scenario

**Scenario 02/10; pattern 003; Beginner.** This is a self-contained offline exercise. An administrative state model separates intended shutdown state from real operational telemetry.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet2/1'` |

**Logic to trace:** Initialize the interface as administratively up. Check the state before and after the operation. Follow the class-specific steps under Pattern 003 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
Ethernet2/1 False
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Changing this Python attribute does not configure a switch; admin state is also not link state.

In [22]:
# Example 0022: An instance method changes state - campus
case = {'interface': 'Ethernet2/1'}

class Interface:
    def __init__(self, name):
        self.name = name
        self.admin_up = True

    def shutdown(self):
        self.admin_up = False
        return self.admin_up

interface = Interface(case["interface"])
assert interface.admin_up is True
assert interface.shutdown() is False
print(interface.name, interface.admin_up)


Ethernet2/1 False


### Example 0023 - An instance method changes state: dc scenario

**Scenario 03/10; pattern 003; Beginner.** This is a self-contained offline exercise. An administrative state model separates intended shutdown state from real operational telemetry.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet3/1'` |

**Logic to trace:** Initialize the interface as administratively up. Check the state before and after the operation. Follow the class-specific steps under Pattern 003 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
Ethernet3/1 False
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Changing this Python attribute does not configure a switch; admin state is also not link state.

In [23]:
# Example 0023: An instance method changes state - dc
case = {'interface': 'Ethernet3/1'}

class Interface:
    def __init__(self, name):
        self.name = name
        self.admin_up = True

    def shutdown(self):
        self.admin_up = False
        return self.admin_up

interface = Interface(case["interface"])
assert interface.admin_up is True
assert interface.shutdown() is False
print(interface.name, interface.admin_up)


Ethernet3/1 False


### Example 0024 - An instance method changes state: wan scenario

**Scenario 04/10; pattern 003; Beginner.** This is a self-contained offline exercise. An administrative state model separates intended shutdown state from real operational telemetry.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet4/1'` |

**Logic to trace:** Initialize the interface as administratively up. Check the state before and after the operation. Follow the class-specific steps under Pattern 003 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
Ethernet4/1 False
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Changing this Python attribute does not configure a switch; admin state is also not link state.

In [24]:
# Example 0024: An instance method changes state - wan
case = {'interface': 'Ethernet4/1'}

class Interface:
    def __init__(self, name):
        self.name = name
        self.admin_up = True

    def shutdown(self):
        self.admin_up = False
        return self.admin_up

interface = Interface(case["interface"])
assert interface.admin_up is True
assert interface.shutdown() is False
print(interface.name, interface.admin_up)


Ethernet4/1 False


### Example 0025 - An instance method changes state: lab scenario

**Scenario 05/10; pattern 003; Beginner.** This is a self-contained offline exercise. An administrative state model separates intended shutdown state from real operational telemetry.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet5/1'` |

**Logic to trace:** Initialize the interface as administratively up. Check the state before and after the operation. Follow the class-specific steps under Pattern 003 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
Ethernet5/1 False
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Changing this Python attribute does not configure a switch; admin state is also not link state.

In [25]:
# Example 0025: An instance method changes state - lab
case = {'interface': 'Ethernet5/1'}

class Interface:
    def __init__(self, name):
        self.name = name
        self.admin_up = True

    def shutdown(self):
        self.admin_up = False
        return self.admin_up

interface = Interface(case["interface"])
assert interface.admin_up is True
assert interface.shutdown() is False
print(interface.name, interface.admin_up)


Ethernet5/1 False


### Example 0026 - An instance method changes state: wireless scenario

**Scenario 06/10; pattern 003; Beginner.** This is a self-contained offline exercise. An administrative state model separates intended shutdown state from real operational telemetry.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet6/1'` |

**Logic to trace:** Initialize the interface as administratively up. Check the state before and after the operation. Follow the class-specific steps under Pattern 003 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
Ethernet6/1 False
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Changing this Python attribute does not configure a switch; admin state is also not link state.

In [26]:
# Example 0026: An instance method changes state - wireless
case = {'interface': 'Ethernet6/1'}

class Interface:
    def __init__(self, name):
        self.name = name
        self.admin_up = True

    def shutdown(self):
        self.admin_up = False
        return self.admin_up

interface = Interface(case["interface"])
assert interface.admin_up is True
assert interface.shutdown() is False
print(interface.name, interface.admin_up)


Ethernet6/1 False


### Example 0027 - An instance method changes state: remote scenario

**Scenario 07/10; pattern 003; Beginner.** This is a self-contained offline exercise. An administrative state model separates intended shutdown state from real operational telemetry.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet7/1'` |

**Logic to trace:** Initialize the interface as administratively up. Check the state before and after the operation. Follow the class-specific steps under Pattern 003 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
Ethernet7/1 False
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Changing this Python attribute does not configure a switch; admin state is also not link state.

In [27]:
# Example 0027: An instance method changes state - remote
case = {'interface': 'Ethernet7/1'}

class Interface:
    def __init__(self, name):
        self.name = name
        self.admin_up = True

    def shutdown(self):
        self.admin_up = False
        return self.admin_up

interface = Interface(case["interface"])
assert interface.admin_up is True
assert interface.shutdown() is False
print(interface.name, interface.admin_up)


Ethernet7/1 False


### Example 0028 - An instance method changes state: core scenario

**Scenario 08/10; pattern 003; Beginner.** This is a self-contained offline exercise. An administrative state model separates intended shutdown state from real operational telemetry.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet8/1'` |

**Logic to trace:** Initialize the interface as administratively up. Check the state before and after the operation. Follow the class-specific steps under Pattern 003 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
Ethernet8/1 False
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Changing this Python attribute does not configure a switch; admin state is also not link state.

In [28]:
# Example 0028: An instance method changes state - core
case = {'interface': 'Ethernet8/1'}

class Interface:
    def __init__(self, name):
        self.name = name
        self.admin_up = True

    def shutdown(self):
        self.admin_up = False
        return self.admin_up

interface = Interface(case["interface"])
assert interface.admin_up is True
assert interface.shutdown() is False
print(interface.name, interface.admin_up)


Ethernet8/1 False


### Example 0029 - An instance method changes state: mgmt scenario

**Scenario 09/10; pattern 003; Beginner.** This is a self-contained offline exercise. An administrative state model separates intended shutdown state from real operational telemetry.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet9/1'` |

**Logic to trace:** Initialize the interface as administratively up. Check the state before and after the operation. Follow the class-specific steps under Pattern 003 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
Ethernet9/1 False
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Changing this Python attribute does not configure a switch; admin state is also not link state.

In [29]:
# Example 0029: An instance method changes state - mgmt
case = {'interface': 'Ethernet9/1'}

class Interface:
    def __init__(self, name):
        self.name = name
        self.admin_up = True

    def shutdown(self):
        self.admin_up = False
        return self.admin_up

interface = Interface(case["interface"])
assert interface.admin_up is True
assert interface.shutdown() is False
print(interface.name, interface.admin_up)


Ethernet9/1 False


### Example 0030 - An instance method changes state: backup scenario

**Scenario 10/10; pattern 003; Beginner.** This is a self-contained offline exercise. An administrative state model separates intended shutdown state from real operational telemetry.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet10/1'` |

**Logic to trace:** Initialize the interface as administratively up. Check the state before and after the operation. Follow the class-specific steps under Pattern 003 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
Ethernet10/1 False
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Changing this Python attribute does not configure a switch; admin state is also not link state.

In [30]:
# Example 0030: An instance method changes state - backup
case = {'interface': 'Ethernet10/1'}

class Interface:
    def __init__(self, name):
        self.name = name
        self.admin_up = True

    def shutdown(self):
        self.admin_up = False
        return self.admin_up

interface = Interface(case["interface"])
assert interface.admin_up is True
assert interface.shutdown() is False
print(interface.name, interface.admin_up)


Ethernet10/1 False


## Pattern 004: Independent objects versus aliases

**Core concept:** Identity and equality are different questions.

**Network-engineering purpose:** A mistaken alias can change the wrong inventory record during a lab exercise.

### Read the implementation in this order
1. Create two independent objects.
2. Bind alias to the first object without copying it.
3. Change a field through alias.
4. Use is for identity and verify the second object is untouched.

**Common mistake / boundary:** The equals sign binds a name; it does not clone an object.

**Practice after the ten worked scenarios:** Draw the three variable names and the two actual objects as a reference diagram.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0031 - Independent objects versus aliases: branch scenario

**Scenario 01/10; pattern 004; Beginner.** This is a self-contained offline exercise. A mistaken alias can change the wrong inventory record during a lab exercise.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Create two independent objects. Use is for identity and verify the second object is untouched. Follow the class-specific steps under Pattern 004 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
branch-edge-01-renamed branch-edge-01
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** The equals sign binds a name; it does not clone an object.

In [31]:
# Example 0031: Independent objects versus aliases - branch
case = {'hostname': 'branch-edge-01'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

first = Device(case["hostname"])
second = Device(case["hostname"])
alias = first
alias.hostname = case["hostname"] + "-renamed"
assert alias is first and second is not first
assert second.hostname == case["hostname"]
print(first.hostname, second.hostname)


branch-edge-01-renamed branch-edge-01


### Example 0032 - Independent objects versus aliases: campus scenario

**Scenario 02/10; pattern 004; Beginner.** This is a self-contained offline exercise. A mistaken alias can change the wrong inventory record during a lab exercise.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Create two independent objects. Use is for identity and verify the second object is untouched. Follow the class-specific steps under Pattern 004 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
campus-access-02-renamed campus-access-02
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** The equals sign binds a name; it does not clone an object.

In [32]:
# Example 0032: Independent objects versus aliases - campus
case = {'hostname': 'campus-access-02'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

first = Device(case["hostname"])
second = Device(case["hostname"])
alias = first
alias.hostname = case["hostname"] + "-renamed"
assert alias is first and second is not first
assert second.hostname == case["hostname"]
print(first.hostname, second.hostname)


campus-access-02-renamed campus-access-02


### Example 0033 - Independent objects versus aliases: dc scenario

**Scenario 03/10; pattern 004; Beginner.** This is a self-contained offline exercise. A mistaken alias can change the wrong inventory record during a lab exercise.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Create two independent objects. Use is for identity and verify the second object is untouched. Follow the class-specific steps under Pattern 004 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
dc-leaf-03-renamed dc-leaf-03
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** The equals sign binds a name; it does not clone an object.

In [33]:
# Example 0033: Independent objects versus aliases - dc
case = {'hostname': 'dc-leaf-03'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

first = Device(case["hostname"])
second = Device(case["hostname"])
alias = first
alias.hostname = case["hostname"] + "-renamed"
assert alias is first and second is not first
assert second.hostname == case["hostname"]
print(first.hostname, second.hostname)


dc-leaf-03-renamed dc-leaf-03


### Example 0034 - Independent objects versus aliases: wan scenario

**Scenario 04/10; pattern 004; Beginner.** This is a self-contained offline exercise. A mistaken alias can change the wrong inventory record during a lab exercise.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Create two independent objects. Use is for identity and verify the second object is untouched. Follow the class-specific steps under Pattern 004 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
wan-border-04-renamed wan-border-04
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** The equals sign binds a name; it does not clone an object.

In [34]:
# Example 0034: Independent objects versus aliases - wan
case = {'hostname': 'wan-border-04'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

first = Device(case["hostname"])
second = Device(case["hostname"])
alias = first
alias.hostname = case["hostname"] + "-renamed"
assert alias is first and second is not first
assert second.hostname == case["hostname"]
print(first.hostname, second.hostname)


wan-border-04-renamed wan-border-04


### Example 0035 - Independent objects versus aliases: lab scenario

**Scenario 05/10; pattern 004; Beginner.** This is a self-contained offline exercise. A mistaken alias can change the wrong inventory record during a lab exercise.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Create two independent objects. Use is for identity and verify the second object is untouched. Follow the class-specific steps under Pattern 004 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
lab-router-05-renamed lab-router-05
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** The equals sign binds a name; it does not clone an object.

In [35]:
# Example 0035: Independent objects versus aliases - lab
case = {'hostname': 'lab-router-05'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

first = Device(case["hostname"])
second = Device(case["hostname"])
alias = first
alias.hostname = case["hostname"] + "-renamed"
assert alias is first and second is not first
assert second.hostname == case["hostname"]
print(first.hostname, second.hostname)


lab-router-05-renamed lab-router-05


### Example 0036 - Independent objects versus aliases: wireless scenario

**Scenario 06/10; pattern 004; Beginner.** This is a self-contained offline exercise. A mistaken alias can change the wrong inventory record during a lab exercise.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Create two independent objects. Use is for identity and verify the second object is untouched. Follow the class-specific steps under Pattern 004 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
wireless-dist-06-renamed wireless-dist-06
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** The equals sign binds a name; it does not clone an object.

In [36]:
# Example 0036: Independent objects versus aliases - wireless
case = {'hostname': 'wireless-dist-06'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

first = Device(case["hostname"])
second = Device(case["hostname"])
alias = first
alias.hostname = case["hostname"] + "-renamed"
assert alias is first and second is not first
assert second.hostname == case["hostname"]
print(first.hostname, second.hostname)


wireless-dist-06-renamed

 wireless-dist-06


### Example 0037 - Independent objects versus aliases: remote scenario

**Scenario 07/10; pattern 004; Beginner.** This is a self-contained offline exercise. A mistaken alias can change the wrong inventory record during a lab exercise.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Create two independent objects. Use is for identity and verify the second object is untouched. Follow the class-specific steps under Pattern 004 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
remote-gateway-07-renamed remote-gateway-07
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** The equals sign binds a name; it does not clone an object.

In [37]:
# Example 0037: Independent objects versus aliases - remote
case = {'hostname': 'remote-gateway-07'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

first = Device(case["hostname"])
second = Device(case["hostname"])
alias = first
alias.hostname = case["hostname"] + "-renamed"
assert alias is first and second is not first
assert second.hostname == case["hostname"]
print(first.hostname, second.hostname)


remote-gateway-07-renamed remote-gateway-07


### Example 0038 - Independent objects versus aliases: core scenario

**Scenario 08/10; pattern 004; Beginner.** This is a self-contained offline exercise. A mistaken alias can change the wrong inventory record during a lab exercise.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Create two independent objects. Use is for identity and verify the second object is untouched. Follow the class-specific steps under Pattern 004 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
core-router-08-renamed core-router-08
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** The equals sign binds a name; it does not clone an object.

In [38]:
# Example 0038: Independent objects versus aliases - core
case = {'hostname': 'core-router-08'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

first = Device(case["hostname"])
second = Device(case["hostname"])
alias = first
alias.hostname = case["hostname"] + "-renamed"
assert alias is first and second is not first
assert second.hostname == case["hostname"]
print(first.hostname, second.hostname)


core-router-08-renamed core-router-08


### Example 0039 - Independent objects versus aliases: mgmt scenario

**Scenario 09/10; pattern 004; Beginner.** This is a self-contained offline exercise. A mistaken alias can change the wrong inventory record during a lab exercise.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Create two independent objects. Use is for identity and verify the second object is untouched. Follow the class-specific steps under Pattern 004 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
mgmt-switch-09-renamed mgmt-switch-09
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** The equals sign binds a name; it does not clone an object.

In [39]:
# Example 0039: Independent objects versus aliases - mgmt
case = {'hostname': 'mgmt-switch-09'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

first = Device(case["hostname"])
second = Device(case["hostname"])
alias = first
alias.hostname = case["hostname"] + "-renamed"
assert alias is first and second is not first
assert second.hostname == case["hostname"]
print(first.hostname, second.hostname)


mgmt-switch-09-renamed mgmt-switch-09


### Example 0040 - Independent objects versus aliases: backup scenario

**Scenario 10/10; pattern 004; Beginner.** This is a self-contained offline exercise. A mistaken alias can change the wrong inventory record during a lab exercise.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Create two independent objects. Use is for identity and verify the second object is untouched. Follow the class-specific steps under Pattern 004 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
backup-edge-10-renamed backup-edge-10
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** The equals sign binds a name; it does not clone an object.

In [40]:
# Example 0040: Independent objects versus aliases - backup
case = {'hostname': 'backup-edge-10'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

first = Device(case["hostname"])
second = Device(case["hostname"])
alias = first
alias.hostname = case["hostname"] + "-renamed"
assert alias is first and second is not first
assert second.hostname == case["hostname"]
print(first.hostname, second.hostname)


backup-edge-10-renamed backup-edge-10


## Pattern 005: Return a calculated result

**Core concept:** An object can compute a value from state without mutating it.

**Network-engineering purpose:** A port-capacity model reports remaining lab capacity and makes its arithmetic testable.

### Read the implementation in this order
1. Store total and used capacity.
2. Compute remaining capacity in a method.
3. Check the arithmetic independently.
4. Verify the query did not change used capacity.

**Common mistake / boundary:** A query that unexpectedly changes state is difficult to reason about and rerun.

**Practice after the ten worked scenarios:** Validate that used is between zero and total, then add a utilization percentage query.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0041 - Return a calculated result: branch scenario

**Scenario 01/10; pattern 005; Beginner.** This is a self-contained offline exercise. A port-capacity model reports remaining lab capacity and makes its arithmetic testable.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `capacity` | `24` |
| `used` | `0` |

**Logic to trace:** Store total and used capacity. Verify the query did not change used capacity. Follow the class-specific steps under Pattern 005 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
free ports: 24
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A query that unexpectedly changes state is difficult to reason about and rerun.

In [41]:
# Example 0041: Return a calculated result - branch
case = {'capacity': 24, 'used': 0}

class PortCapacity:
    def __init__(self, total, used):
        self.total = total
        self.used = used

    def remaining(self):
        return self.total - self.used

ports = PortCapacity(case["capacity"], case["used"])
assert ports.remaining() == case["capacity"] - case["used"]
assert ports.used == case["used"]
print("free ports:", ports.remaining())


free ports: 24


### Example 0042 - Return a calculated result: campus scenario

**Scenario 02/10; pattern 005; Beginner.** This is a self-contained offline exercise. A port-capacity model reports remaining lab capacity and makes its arithmetic testable.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `capacity` | `48` |
| `used` | `48` |

**Logic to trace:** Store total and used capacity. Verify the query did not change used capacity. Follow the class-specific steps under Pattern 005 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
free ports: 0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A query that unexpectedly changes state is difficult to reason about and rerun.

In [42]:
# Example 0042: Return a calculated result - campus
case = {'capacity': 48, 'used': 48}

class PortCapacity:
    def __init__(self, total, used):
        self.total = total
        self.used = used

    def remaining(self):
        return self.total - self.used

ports = PortCapacity(case["capacity"], case["used"])
assert ports.remaining() == case["capacity"] - case["used"]
assert ports.used == case["used"]
print("free ports:", ports.remaining())


free ports:

 0


### Example 0043 - Return a calculated result: dc scenario

**Scenario 03/10; pattern 005; Beginner.** This is a self-contained offline exercise. A port-capacity model reports remaining lab capacity and makes its arithmetic testable.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `capacity` | `32` |
| `used` | `16` |

**Logic to trace:** Store total and used capacity. Verify the query did not change used capacity. Follow the class-specific steps under Pattern 005 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
free ports: 16
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A query that unexpectedly changes state is difficult to reason about and rerun.

In [43]:
# Example 0043: Return a calculated result - dc
case = {'capacity': 32, 'used': 16}

class PortCapacity:
    def __init__(self, total, used):
        self.total = total
        self.used = used

    def remaining(self):
        return self.total - self.used

ports = PortCapacity(case["capacity"], case["used"])
assert ports.remaining() == case["capacity"] - case["used"]
assert ports.used == case["used"]
print("free ports:", ports.remaining())


free ports: 16


### Example 0044 - Return a calculated result: wan scenario

**Scenario 04/10; pattern 005; Beginner.** This is a self-contained offline exercise. A port-capacity model reports remaining lab capacity and makes its arithmetic testable.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `capacity` | `16` |
| `used` | `15` |

**Logic to trace:** Store total and used capacity. Verify the query did not change used capacity. Follow the class-specific steps under Pattern 005 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
free ports: 1
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A query that unexpectedly changes state is difficult to reason about and rerun.

In [44]:
# Example 0044: Return a calculated result - wan
case = {'capacity': 16, 'used': 15}

class PortCapacity:
    def __init__(self, total, used):
        self.total = total
        self.used = used

    def remaining(self):
        return self.total - self.used

ports = PortCapacity(case["capacity"], case["used"])
assert ports.remaining() == case["capacity"] - case["used"]
assert ports.used == case["used"]
print("free ports:", ports.remaining())


free ports: 1


### Example 0045 - Return a calculated result: lab scenario

**Scenario 05/10; pattern 005; Beginner.** This is a self-contained offline exercise. A port-capacity model reports remaining lab capacity and makes its arithmetic testable.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `capacity` | `8` |
| `used` | `1` |

**Logic to trace:** Store total and used capacity. Verify the query did not change used capacity. Follow the class-specific steps under Pattern 005 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
free ports: 7
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A query that unexpectedly changes state is difficult to reason about and rerun.

In [45]:
# Example 0045: Return a calculated result - lab
case = {'capacity': 8, 'used': 1}

class PortCapacity:
    def __init__(self, total, used):
        self.total = total
        self.used = used

    def remaining(self):
        return self.total - self.used

ports = PortCapacity(case["capacity"], case["used"])
assert ports.remaining() == case["capacity"] - case["used"]
assert ports.used == case["used"]
print("free ports:", ports.remaining())


free ports: 7


### Example 0046 - Return a calculated result: wireless scenario

**Scenario 06/10; pattern 005; Beginner.** This is a self-contained offline exercise. A port-capacity model reports remaining lab capacity and makes its arithmetic testable.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `capacity` | `64` |
| `used` | `32` |

**Logic to trace:** Store total and used capacity. Verify the query did not change used capacity. Follow the class-specific steps under Pattern 005 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
free ports: 32
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A query that unexpectedly changes state is difficult to reason about and rerun.

In [46]:
# Example 0046: Return a calculated result - wireless
case = {'capacity': 64, 'used': 32}

class PortCapacity:
    def __init__(self, total, used):
        self.total = total
        self.used = used

    def remaining(self):
        return self.total - self.used

ports = PortCapacity(case["capacity"], case["used"])
assert ports.remaining() == case["capacity"] - case["used"]
assert ports.used == case["used"]
print("free ports:", ports.remaining())


free ports: 32


### Example 0047 - Return a calculated result: remote scenario

**Scenario 07/10; pattern 005; Beginner.** This is a self-contained offline exercise. A port-capacity model reports remaining lab capacity and makes its arithmetic testable.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `capacity` | `12` |
| `used` | `12` |

**Logic to trace:** Store total and used capacity. Verify the query did not change used capacity. Follow the class-specific steps under Pattern 005 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
free ports: 0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A query that unexpectedly changes state is difficult to reason about and rerun.

In [47]:
# Example 0047: Return a calculated result - remote
case = {'capacity': 12, 'used': 12}

class PortCapacity:
    def __init__(self, total, used):
        self.total = total
        self.used = used

    def remaining(self):
        return self.total - self.used

ports = PortCapacity(case["capacity"], case["used"])
assert ports.remaining() == case["capacity"] - case["used"]
assert ports.used == case["used"]
print("free ports:", ports.remaining())


free ports: 0


### Example 0048 - Return a calculated result: core scenario

**Scenario 08/10; pattern 005; Beginner.** This is a self-contained offline exercise. A port-capacity model reports remaining lab capacity and makes its arithmetic testable.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `capacity` | `96` |
| `used` | `1` |

**Logic to trace:** Store total and used capacity. Verify the query did not change used capacity. Follow the class-specific steps under Pattern 005 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
free ports: 95
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A query that unexpectedly changes state is difficult to reason about and rerun.

In [48]:
# Example 0048: Return a calculated result - core
case = {'capacity': 96, 'used': 1}

class PortCapacity:
    def __init__(self, total, used):
        self.total = total
        self.used = used

    def remaining(self):
        return self.total - self.used

ports = PortCapacity(case["capacity"], case["used"])
assert ports.remaining() == case["capacity"] - case["used"]
assert ports.used == case["used"]
print("free ports:", ports.remaining())


free ports: 95


### Example 0049 - Return a calculated result: mgmt scenario

**Scenario 09/10; pattern 005; Beginner.** This is a self-contained offline exercise. A port-capacity model reports remaining lab capacity and makes its arithmetic testable.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `capacity` | `48` |
| `used` | `47` |

**Logic to trace:** Store total and used capacity. Verify the query did not change used capacity. Follow the class-specific steps under Pattern 005 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
free ports: 1
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A query that unexpectedly changes state is difficult to reason about and rerun.

In [49]:
# Example 0049: Return a calculated result - mgmt
case = {'capacity': 48, 'used': 47}

class PortCapacity:
    def __init__(self, total, used):
        self.total = total
        self.used = used

    def remaining(self):
        return self.total - self.used

ports = PortCapacity(case["capacity"], case["used"])
assert ports.remaining() == case["capacity"] - case["used"]
assert ports.used == case["used"]
print("free ports:", ports.remaining())


free ports: 1


### Example 0050 - Return a calculated result: backup scenario

**Scenario 10/10; pattern 005; Beginner.** This is a self-contained offline exercise. A port-capacity model reports remaining lab capacity and makes its arithmetic testable.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `capacity` | `24` |
| `used` | `12` |

**Logic to trace:** Store total and used capacity. Verify the query did not change used capacity. Follow the class-specific steps under Pattern 005 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
free ports: 12
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A query that unexpectedly changes state is difficult to reason about and rerun.

In [50]:
# Example 0050: Return a calculated result - backup
case = {'capacity': 24, 'used': 12}

class PortCapacity:
    def __init__(self, total, used):
        self.total = total
        self.used = used

    def remaining(self):
        return self.total - self.used

ports = PortCapacity(case["capacity"], case["used"])
assert ports.remaining() == case["capacity"] - case["used"]
assert ports.used == case["used"]
print("free ports:", ports.remaining())


free ports: 12


## Check your understanding: Classes, objects, self, and constructors

**You should now be able to:** Create a class, trace initialization, distinguish identity from value, and choose queries versus state changes.

1. Explain which object owns each piece of state and which operations can change it.
2. Walk through a normal result and an invalid-input or failure path without running the code.
3. Name one invariant, one useful assertion, and one production assumption not proved by the fake.
4. Implement one extension prompt, including a test that would fail if your new behavior were wrong.
5. Explain when a function or a dataclass would be simpler than another subclass or framework.

**Do not confuse:** a successful local assertion with real device compatibility; missing data with a healthy result; underscore naming with access control; type annotations with runtime validation; or local rollback with a multi-device transaction.

## Official reference reading
- `https://docs.python.org/3.13/tutorial/classes.html`
- `https://docs.python.org/3.13/reference/datamodel.html`

References describe Python contracts. They do not certify any vendor command or network policy.